ASSOCIATION RULE MINING

In [1]:
import pandas as pd
import numpy as np

Load processed data

In [2]:

df = pd.read_csv("../data/processed_features.csv")

print("Shape:", df.shape)

display(df.head())

Shape: (3600, 19)


,timestamp,caller,callee,answered,previous_calls_between,previous_answer_rate_between,time_since_last_call,time_since_last_call_hours,time_since_last_call_days,receiver_previous_call_count,receiver_previous_answer_rate,recent_sms_count_48h,time_since_last_sms,time_since_last_sms_hours,time_since_last_sms_days,recent_bluetooth_count_48h,hour_of_day,day_of_week,weekend
0,184,300,301,1,0,NaN,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,0,0.051111,Sunday,1
1,3920,512,299,1,0,NaN,NaN,NaN,NaN,0,NaN,7,2.0,0.000556,0.000023,0,1.088889,Sunday,1
2,5623,301,300,1,1,1.0,5439.0,1.510833,0.062951,0,NaN,1,15.0,0.004167,0.000174,0,1.561944,Sunday,1
3,9252,401,457,0,0,NaN,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,0,2.570000,Sunday,1
4,15466,512,0,1,0,NaN,NaN,NaN,NaN,0,NaN,28,130.0,0.036111,0.001505,0,4.296111,Sunday,1


In [3]:
# Check whether the pattern remains with more recorded history

comparison = []

for minimum_history in [0, 5]:

    subset = df.loc[
        df["receiver_previous_call_count"] >= minimum_history
    ].copy()

    # Same definition as in the association-rule analysis:
    # low receiver answer rate = 0–50%
    low_rate = subset["receiver_previous_answer_rate"].between(
        0, 0.5, inclusive="both"
    )

    missed = subset["answered"].eq(0)

    number_of_calls = len(subset)
    pattern_calls = int(low_rate.sum())
    pattern_and_missed = int((low_rate & missed).sum())

    baseline = (
        missed.mean()
        if number_of_calls > 0
        else np.nan
    )

    confidence = (
        pattern_and_missed / pattern_calls
        if pattern_calls > 0
        else np.nan
    )

    support = (
        pattern_and_missed / number_of_calls
        if number_of_calls > 0
        else np.nan
    )

    lift = (
        confidence / baseline
        if baseline > 0
        else np.nan
    )

    comparison.append({
        "minimum_previous_calls": minimum_history,
        "total_calls": number_of_calls,
        "calls_with_low_answer_rate": pattern_calls,
        "missed_calls_with_low_answer_rate": pattern_and_missed,
        "baseline_missed_pct": baseline * 100,
        "support_pct": support * 100,
        "confidence_pct": confidence * 100,
        "lift": lift
    })

history_check = pd.DataFrame(comparison)

display(history_check.round(3))

,minimum_previous_calls,total_calls,calls_with_low_answer_rate,missed_calls_with_low_answer_rate,baseline_missed_pct,support_pct,confidence_pct,lift
0,0,3600,195,73,10.167,2.028,37.436,3.682
1,5,1925,47,24,9.506,1.247,51.064,5.371


Select variables for association rule mining

In [4]:
arm_data = df[
    [
        "answered",
        "previous_calls_between",
        "previous_answer_rate_between",
        "time_since_last_call_hours",
        "receiver_previous_answer_rate",
        "recent_sms_count_48h",
        "time_since_last_sms_hours",
        "recent_bluetooth_count_48h",
        "hour_of_day",
        "day_of_week",
        "weekend"
    ]
].copy()

display(arm_data.head())

,answered,previous_calls_between,previous_answer_rate_between,time_since_last_call_hours,receiver_previous_answer_rate,recent_sms_count_48h,time_since_last_sms_hours,recent_bluetooth_count_48h,hour_of_day,day_of_week,weekend
0,1,0,NaN,NaN,NaN,0,NaN,0,0.051111,Sunday,1
1,1,0,NaN,NaN,NaN,7,0.000556,0,1.088889,Sunday,1
2,1,1,1.0,1.510833,NaN,1,0.004167,0,1.561944,Sunday,1
3,0,0,NaN,NaN,NaN,0,NaN,0,2.570000,Sunday,1
4,1,0,NaN,NaN,NaN,28,0.036111,0,4.296111,Sunday,1


Inspect numerical feature disturbutions

In [5]:
numerical_features = [
    "previous_calls_between",
    "previous_answer_rate_between",
    "time_since_last_call_hours",
    "receiver_previous_answer_rate",
    "recent_sms_count_48h",
    "time_since_last_sms_hours",
    "recent_bluetooth_count_48h",
    "hour_of_day"
]

display(
    arm_data[numerical_features].describe().T
)

,count,mean,std,min,25%,50%,75%,max
previous_calls_between,3600.0,11.190556,17.788774,0.000000,1.000000,4.000000,13.000000,129.000000
previous_answer_rate_between,2979.0,0.886047,0.216842,0.000000,0.851852,1.000000,1.000000,1.000000
time_since_last_call_hours,2979.0,34.824505,76.898498,0.000556,0.047917,0.981667,26.197361,628.075556
receiver_previous_answer_rate,3120.0,0.900757,0.200582,0.000000,0.900000,1.000000,1.000000,1.000000
recent_sms_count_48h,3600.0,9.184444,21.139224,0.000000,0.000000,2.000000,9.000000,326.000000
time_since_last_sms_hours,2857.0,47.815917,88.811176,0.000278,0.357222,5.438056,50.251667,639.529444
recent_bluetooth_count_48h,3600.0,60.740000,99.540884,0.000000,0.000000,12.000000,82.000000,570.000000
hour_of_day,3600.0,14.972728,4.819655,0.004722,12.067153,15.519861,18.193056,23.997222


In [6]:
# Convert numerical features to discrete values
arm_discrete = arm_data.copy()

# Call outcome
arm_discrete["call_outcome"] = arm_discrete["answered"].map({
    1: "answered",
    0: "missed"
})

In [7]:
arm_discrete["previous_calls_category"] = pd.cut(
    arm_discrete["previous_calls_between"],
    bins=[-1, 0, 3, 13, np.inf],
    labels=[
        "no_previous_calls",
        "low_previous_calls",
        "medium_previous_calls",
        "high_previous_calls"
    ]
)

In [8]:
arm_discrete["previous_answer_rate_category"] = pd.cut(
    arm_discrete["previous_answer_rate_between"],
    bins=[-0.01, 0.5, 0.8, 0.999999, 1.0],
    labels=[
        "low_answer_rate",
        "medium_answer_rate",
        "high_answer_rate",
        "perfect_answer_rate"
    ],
    include_lowest=True
)

arm_discrete["previous_answer_rate_category"] = (
    arm_discrete["previous_answer_rate_category"]
    .cat.add_categories("no_previous_call_history")
    .fillna("no_previous_call_history")
)

In [9]:
#Redo time limits for time since last call to be more meaningful
arm_discrete["time_since_last_call_category"] = pd.cut(
    arm_discrete["time_since_last_call_hours"],
    bins=[0, 1, 24, 72, np.inf],
    labels=[
        "within_1h",
        "1h_to_24h",
        "1d_to_3d",
        "more_than_3d"
    ],
    include_lowest=True
)

arm_discrete["time_since_last_call_category"] = (
    arm_discrete["time_since_last_call_category"]
    .cat.add_categories("no_previous_call")
    .fillna("no_previous_call")
)

In [10]:
arm_discrete["receiver_answer_rate_category"] = pd.cut(
    arm_discrete["receiver_previous_answer_rate"],
    bins=[-0.01, 0.5, 0.8, 0.999999, 1.0],
    labels=[
        "low_receiver_answer_rate",
        "medium_receiver_answer_rate",
        "high_receiver_answer_rate",
        "perfect_receiver_answer_rate"
    ],
    include_lowest=True
)

arm_discrete["receiver_answer_rate_category"] = (
    arm_discrete["receiver_answer_rate_category"]
    .cat.add_categories("no_receiver_history")
    .fillna("no_receiver_history")
)

In [11]:
arm_discrete["recent_sms_category"] = pd.cut(
    arm_discrete["recent_sms_count_48h"],
    bins=[-1, 0, 2, 9, np.inf],
    labels=[
        "no_recent_sms",
        "low_recent_sms",
        "medium_recent_sms",
        "high_recent_sms"
    ]
)

In [12]:
arm_discrete["time_since_last_sms_category"] = pd.cut(
    arm_discrete["time_since_last_sms_hours"],
    bins=[0, 6, 24, 72, np.inf],
    labels=[
        "within_6h",
        "6h_to_24h",
        "1d_to_3d",
        "more_than_3d"
    ],
    include_lowest=True
)

arm_discrete["time_since_last_sms_category"] = (
    arm_discrete["time_since_last_sms_category"]
    .cat.add_categories("no_previous_sms")
    .fillna("no_previous_sms")
)

In [13]:
arm_discrete["bluetooth_category"] = pd.cut(
    arm_discrete["recent_bluetooth_count_48h"],
    bins=[-1, 0, 12, 82, np.inf],
    labels=[
        "no_recent_proximity",
        "low_recent_proximity",
        "medium_recent_proximity",
        "high_recent_proximity"
    ]
)

In [14]:
arm_discrete["time_of_day_category"] = pd.cut(
    arm_discrete["hour_of_day"],
    bins=[0, 6, 12, 18, 24],
    labels=[
        "night",
        "morning",
        "afternoon",
        "evening"
    ],
    include_lowest=True,
    right=False
)

In [15]:
# Control
categorical_features = [
    "call_outcome",
    "previous_calls_category",
    "previous_answer_rate_category",
    "time_since_last_call_category",
    "receiver_answer_rate_category",
    "recent_sms_category",
    "time_since_last_sms_category",
    "bluetooth_category",
    "time_of_day_category",
    "day_of_week",
    "weekend"
]

for column in categorical_features:
    print(f"\n--- {column} ---")
    print(arm_discrete[column].value_counts(dropna=False))


--- call_outcome ---
call_outcome
answered    3234
missed       366
Name: count, dtype: int64

--- previous_calls_category ---
previous_calls_category
medium_previous_calls    1120
low_previous_calls        985
high_previous_calls       874
no_previous_calls         621
Name: count, dtype: int64

--- previous_answer_rate_category ---
previous_answer_rate_category
perfect_answer_rate         1685
high_answer_rate             676
no_previous_call_history     621
medium_answer_rate           405
low_answer_rate              213
Name: count, dtype: int64

--- time_since_last_call_category ---
time_since_last_call_category
within_1h           1494
1h_to_24h            685
no_previous_call     621
more_than_3d         433
1d_to_3d             367
Name: count, dtype: int64

--- receiver_answer_rate_category ---
receiver_answer_rate_category
perfect_receiver_answer_rate    1941
high_receiver_answer_rate        592
no_receiver_history              480
medium_receiver_answer_rate      392
low_r

Prepare transactions for association rule mining

In [16]:
transaction_data = arm_discrete[
    [
        "call_outcome",
        "previous_calls_category",
        "previous_answer_rate_category",
        "time_since_last_call_category",
        "receiver_answer_rate_category",
        "recent_sms_category",
        "time_since_last_sms_category",
        "bluetooth_category",
        "time_of_day_category",
        "day_of_week",
        "weekend"
    ]
].copy()

display(transaction_data.head())

,call_outcome,previous_calls_category,previous_answer_rate_category,time_since_last_call_category,receiver_answer_rate_category,recent_sms_category,time_since_last_sms_category,bluetooth_category,time_of_day_category,day_of_week,weekend
0,answered,no_previous_calls,no_previous_call_history,no_previous_call,no_receiver_history,no_recent_sms,no_previous_sms,no_recent_proximity,night,Sunday,1
1,answered,no_previous_calls,no_previous_call_history,no_previous_call,no_receiver_history,medium_recent_sms,within_6h,no_recent_proximity,night,Sunday,1
2,answered,low_previous_calls,perfect_answer_rate,1h_to_24h,no_receiver_history,low_recent_sms,within_6h,no_recent_proximity,night,Sunday,1
3,missed,no_previous_calls,no_previous_call_history,no_previous_call,no_receiver_history,no_recent_sms,no_previous_sms,no_recent_proximity,night,Sunday,1
4,answered,no_previous_calls,no_previous_call_history,no_previous_call,no_receiver_history,high_recent_sms,within_6h,no_recent_proximity,night,Sunday,1


In [17]:
# Add feature names to category values

transaction_data["call_outcome"] = (
    "call_outcome=" + transaction_data["call_outcome"].astype(str)
)

transaction_data["previous_calls_category"] = (
    "previous_calls=" + transaction_data["previous_calls_category"].astype(str)
)

transaction_data["previous_answer_rate_category"] = (
    "previous_answer_rate=" +
    transaction_data["previous_answer_rate_category"].astype(str)
)

transaction_data["time_since_last_call_category"] = (
    "time_since_last_call=" +
    transaction_data["time_since_last_call_category"].astype(str)
)

transaction_data["receiver_answer_rate_category"] = (
    "receiver_answer_rate=" +
    transaction_data["receiver_answer_rate_category"].astype(str)
)

transaction_data["recent_sms_category"] = (
    "recent_sms=" + transaction_data["recent_sms_category"].astype(str)
)

transaction_data["time_since_last_sms_category"] = (
    "time_since_last_sms=" +
    transaction_data["time_since_last_sms_category"].astype(str)
)

transaction_data["bluetooth_category"] = (
    "bluetooth=" + transaction_data["bluetooth_category"].astype(str)
)

transaction_data["time_of_day_category"] = (
    "time_of_day=" + transaction_data["time_of_day_category"].astype(str)
)

transaction_data["day_of_week"] = (
    "day_of_week=" + transaction_data["day_of_week"].astype(str)
)

transaction_data["weekend"] = transaction_data["weekend"].map({
    0: "weekend=no",
    1: "weekend=yes"
})

display(transaction_data.head())

,call_outcome,previous_calls_category,previous_answer_rate_category,time_since_last_call_category,receiver_answer_rate_category,recent_sms_category,time_since_last_sms_category,bluetooth_category,time_of_day_category,day_of_week,weekend
0,call_outcome=answered,previous_calls=no_previous_calls,previous_answer_rate=no_previous_call_history,time_since_last_call=no_previous_call,receiver_answer_rate=no_receiver_history,recent_sms=no_recent_sms,time_since_last_sms=no_previous_sms,bluetooth=no_recent_proximity,time_of_day=night,day_of_week=Sunday,weekend=yes
1,call_outcome=answered,previous_calls=no_previous_calls,previous_answer_rate=no_previous_call_history,time_since_last_call=no_previous_call,receiver_answer_rate=no_receiver_history,recent_sms=medium_recent_sms,time_since_last_sms=within_6h,bluetooth=no_recent_proximity,time_of_day=night,day_of_week=Sunday,weekend=yes
2,call_outcome=answered,previous_calls=low_previous_calls,previous_answer_rate=perfect_answer_rate,time_since_last_call=1h_to_24h,receiver_answer_rate=no_receiver_history,recent_sms=low_recent_sms,time_since_last_sms=within_6h,bluetooth=no_recent_proximity,time_of_day=night,day_of_week=Sunday,weekend=yes
3,call_outcome=missed,previous_calls=no_previous_calls,previous_answer_rate=no_previous_call_history,time_since_last_call=no_previous_call,receiver_answer_rate=no_receiver_history,recent_sms=no_recent_sms,time_since_last_sms=no_previous_sms,bluetooth=no_recent_proximity,time_of_day=night,day_of_week=Sunday,weekend=yes
4,call_outcome=answered,previous_calls=no_previous_calls,previous_answer_rate=no_previous_call_history,time_since_last_call=no_previous_call,receiver_answer_rate=no_receiver_history,recent_sms=high_recent_sms,time_since_last_sms=within_6h,bluetooth=no_recent_proximity,time_of_day=night,day_of_week=Sunday,weekend=yes


One-hot encode transactions

In [18]:
transactions_encoded = pd.get_dummies(
    transaction_data,
    prefix="",
    prefix_sep=""
).astype(bool)

print("Shape:", transactions_encoded.shape)

display(transactions_encoded.head())

Shape: (3600, 47)


,call_outcome=answered,call_outcome=missed,previous_calls=high_previous_calls,previous_calls=low_previous_calls,previous_calls=medium_previous_calls,previous_calls=no_previous_calls,previous_answer_rate=high_answer_rate,previous_answer_rate=low_answer_rate,previous_answer_rate=medium_answer_rate,previous_answer_rate=no_previous_call_history,...,time_of_day=night,day_of_week=Friday,day_of_week=Monday,day_of_week=Saturday,day_of_week=Sunday,day_of_week=Thursday,day_of_week=Tuesday,day_of_week=Wednesday,weekend=no,weekend=yes
0,True,False,False,False,False,True,False,False,False,True,...,True,False,False,False,True,False,False,False,False,True
1,True,False,False,False,False,True,False,False,False,True,...,True,False,False,False,True,False,False,False,False,True
2,True,False,False,True,False,False,False,False,False,False,...,True,False,False,False,True,False,False,False,False,True
3,False,True,False,False,False,True,False,False,False,True,...,True,False,False,False,True,False,False,False,False,True
4,True,False,False,False,False,True,False,False,False,True,...,True,False,False,False,True,False,False,False,False,True


In [19]:
print("Number of transactions:", len(transactions_encoded))
print("Number of unique items:", transactions_encoded.shape[1])

print("\nItems:")
for item in transactions_encoded.columns:
    print(item)

Number of transactions: 3600
Number of unique items: 47

Items:
call_outcome=answered
call_outcome=missed
previous_calls=high_previous_calls
previous_calls=low_previous_calls
previous_calls=medium_previous_calls
previous_calls=no_previous_calls
previous_answer_rate=high_answer_rate
previous_answer_rate=low_answer_rate
previous_answer_rate=medium_answer_rate
previous_answer_rate=no_previous_call_history
previous_answer_rate=perfect_answer_rate
time_since_last_call=1d_to_3d
time_since_last_call=1h_to_24h
time_since_last_call=more_than_3d
time_since_last_call=no_previous_call
time_since_last_call=within_1h
receiver_answer_rate=high_receiver_answer_rate
receiver_answer_rate=low_receiver_answer_rate
receiver_answer_rate=medium_receiver_answer_rate
receiver_answer_rate=no_receiver_history
receiver_answer_rate=perfect_receiver_answer_rate
recent_sms=high_recent_sms
recent_sms=low_recent_sms
recent_sms=medium_recent_sms
recent_sms=no_recent_sms
time_since_last_sms=1d_to_3d
time_since_last_sms=

In [20]:
from mlxtend.frequent_patterns import apriori, association_rules

In [21]:
# A lower minimum support is used because missed calls
# represent only about 10% of all observations.
#
# min_support = 0.02 corresponds to at least 72 calls.
# max_len = 4 limits itemsets to manageable combinations.

frequent_itemsets = apriori(
    transactions_encoded,
    min_support=0.02,
    use_colnames=True,
    max_len=4
)

frequent_itemsets = frequent_itemsets.sort_values(
    "support",
    ascending=False
).reset_index(drop=True)

print("Number of frequent itemsets:", len(frequent_itemsets))

Number of frequent itemsets: 6243


In [22]:
missed_frequent_itemsets = frequent_itemsets[
    frequent_itemsets["itemsets"].apply(
        lambda x: "call_outcome=missed" in x
    )
]

print(
    "Frequent itemsets containing missed outcome:",
    len(missed_frequent_itemsets)
)

display(
    missed_frequent_itemsets[
        ["support", "itemsets"]
    ].head(20)
)

Frequent itemsets containing missed outcome: 44


,support,itemsets
433,0.101667,(call_outcome=missed)
797,0.076389,"(weekend=no, call_outcome=missed)"
1708,0.049444,"(bluetooth=no_recent_proximity, call_outcome=m..."
1813,0.047778,"(time_of_day=afternoon, call_outcome=missed)"
1879,0.046944,"(call_outcome=missed, recent_sms=no_recent_sms)"
2303,0.041389,"(call_outcome=missed, time_since_last_call=wit..."
2620,0.038056,"(time_of_day=afternoon, weekend=no, call_outco..."
2936,0.035000,"(bluetooth=no_recent_proximity, weekend=no, ca..."
2954,0.035000,"(time_since_last_sms=within_6h, call_outcome=m..."
3119,0.033611,"(previous_calls=medium_previous_calls, call_ou..."


Generate association rules

In [23]:
# Use lift instead of confidence as the initial threshold.
# This is important because missed calls are much less common
# than answered calls.

rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

print("Number of association rules:", len(rules))

display(
    rules[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20)
)

Number of association rules: 45750


,antecedents,consequents,support,confidence,lift
0,(call_outcome=answered),(weekend=no),0.692500,0.770872,1.002579
1,(weekend=no),(call_outcome=answered),0.692500,0.900650,1.002579
2,(call_outcome=answered),(receiver_answer_rate=perfect_receiver_answer_...,0.506944,0.564317,1.046646
3,(receiver_answer_rate=perfect_receiver_answer_...,(call_outcome=answered),0.506944,0.940237,1.046646
4,(call_outcome=answered),(time_of_day=afternoon),0.445556,0.495980,1.005365
5,(time_of_day=afternoon),(call_outcome=answered),0.445556,0.903153,1.005365
6,(call_outcome=answered),(previous_answer_rate=perfect_answer_rate),0.440000,0.489796,1.046448
7,(previous_answer_rate=perfect_answer_rate),(call_outcome=answered),0.440000,0.940059,1.046448
8,(weekend=no),(receiver_answer_rate=perfect_receiver_answer_...,0.418333,0.544075,1.009104
9,(receiver_answer_rate=perfect_receiver_answer_...,(weekend=no),0.418333,0.775889,1.009104


In [24]:
answered_item = frozenset(["call_outcome=answered"])
missed_item = frozenset(["call_outcome=missed"])

outcome_rules = rules[
    rules["consequents"].isin([
        answered_item,
        missed_item
    ])
].copy()

print(
    "Number of rules with exactly one call outcome as consequent:",
    len(outcome_rules)
)

Number of rules with exactly one call outcome as consequent: 1382


In [25]:
answered_rules = outcome_rules[
    outcome_rules["consequents"] == answered_item
].copy()

missed_rules = outcome_rules[
    outcome_rules["consequents"] == missed_item
].copy()

print("Answered rules:", len(answered_rules))
print("Missed rules:", len(missed_rules))

Answered rules: 1356
Missed rules: 26


In [26]:
answered_rules = answered_rules.sort_values(
    ["lift", "support"],
    ascending=[False, False]
).reset_index(drop=True)

missed_rules = missed_rules.sort_values(
    ["lift", "support"],
    ascending=[False, False]
).reset_index(drop=True)

In [27]:
display(
    answered_rules[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20)
)

,antecedents,consequents,support,confidence,lift
0,"(day_of_week=Tuesday, previous_calls=medium_pr...",(call_outcome=answered),0.026389,1.000000,1.113173
1,"(day_of_week=Tuesday, previous_calls=medium_pr...",(call_outcome=answered),0.024722,1.000000,1.113173
2,"(weekend=yes, recent_sms=high_recent_sms, prev...",(call_outcome=answered),0.020833,1.000000,1.113173
3,"(receiver_answer_rate=no_receiver_history, tim...",(call_outcome=answered),0.020556,1.000000,1.113173
4,(receiver_answer_rate=perfect_receiver_answer_...,(call_outcome=answered),0.036111,0.992366,1.104675
5,"(recent_sms=high_recent_sms, previous_answer_r...",(call_outcome=answered),0.031667,0.991304,1.103493
6,"(time_since_last_call=1h_to_24h, recent_sms=no...",(call_outcome=answered),0.030833,0.991071,1.103234
7,"(previous_answer_rate=perfect_answer_rate, rec...",(call_outcome=answered),0.029444,0.990654,1.102769
8,"(previous_calls=high_previous_calls, day_of_we...",(call_outcome=answered),0.027222,0.989899,1.101928
9,"(day_of_week=Wednesday, receiver_answer_rate=p...",(call_outcome=answered),0.027222,0.989899,1.101928


In [28]:
display(
    missed_rules[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(30)
)

,antecedents,consequents,support,confidence,lift
0,(receiver_answer_rate=low_receiver_answer_rate),(call_outcome=missed),0.020278,0.374359,3.682219
1,(receiver_answer_rate=medium_receiver_answer_r...,(call_outcome=missed),0.021944,0.201531,1.982268
2,(time_since_last_sms=no_previous_sms),(call_outcome=missed),0.028889,0.139973,1.376784
3,"(time_since_last_sms=no_previous_sms, recent_s...",(call_outcome=missed),0.028889,0.139973,1.376784
4,(time_of_day=morning),(call_outcome=missed),0.025278,0.130747,1.286037
5,(previous_answer_rate=no_previous_call_history...,(call_outcome=missed),0.021667,0.125604,1.235448
6,"(time_since_last_call=no_previous_call, previo...",(call_outcome=missed),0.021667,0.125604,1.235448
7,(previous_answer_rate=no_previous_call_history...,(call_outcome=missed),0.021667,0.125604,1.235448
8,(time_since_last_call=no_previous_call),(call_outcome=missed),0.021667,0.125604,1.235448
9,(previous_answer_rate=no_previous_call_history...,(call_outcome=missed),0.021667,0.125604,1.235448


In [29]:
print("Total outcome rules:", len(outcome_rules))
print("Answered rules:", len(answered_rules))
print("Missed rules:", len(missed_rules))

Total outcome rules: 1382
Answered rules: 1356
Missed rules: 26


In [30]:
# Sort outcome-related rules by lift

outcome_rules = outcome_rules.sort_values(
    "lift",
    ascending=False
).reset_index(drop=True)

display(
    outcome_rules[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(30)
)

,antecedents,consequents,support,confidence,lift
0,(receiver_answer_rate=low_receiver_answer_rate),(call_outcome=missed),0.020278,0.374359,3.682219
1,(receiver_answer_rate=medium_receiver_answer_r...,(call_outcome=missed),0.021944,0.201531,1.982268
2,"(time_since_last_sms=no_previous_sms, recent_s...",(call_outcome=missed),0.028889,0.139973,1.376784
3,(time_since_last_sms=no_previous_sms),(call_outcome=missed),0.028889,0.139973,1.376784
4,(time_of_day=morning),(call_outcome=missed),0.025278,0.130747,1.286037
5,(previous_answer_rate=no_previous_call_history...,(call_outcome=missed),0.021667,0.125604,1.235448
6,(time_since_last_call=no_previous_call),(call_outcome=missed),0.021667,0.125604,1.235448
7,(previous_answer_rate=no_previous_call_history...,(call_outcome=missed),0.021667,0.125604,1.235448
8,(previous_answer_rate=no_previous_call_history...,(call_outcome=missed),0.021667,0.125604,1.235448
9,(previous_calls=no_previous_calls),(call_outcome=missed),0.021667,0.125604,1.235448


In [31]:
filtered_answered_rules = answered_rules[
    (answered_rules["support"] >= 0.02)
    & (answered_rules["confidence"] >= 0.90)
    & (answered_rules["lift"] >= 1.05)
].copy()

filtered_missed_rules = missed_rules[
    (missed_rules["support"] >= 0.02)
    & (missed_rules["confidence"] >= 0.15)
    & (missed_rules["lift"] >= 1.5)
].copy()

print("Filtered answered rules:", len(filtered_answered_rules))
print("Filtered missed rules:", len(filtered_missed_rules))

Filtered answered rules: 435
Filtered missed rules: 2


In [32]:
filtered_answered_rules = filtered_answered_rules.sort_values(
    ["lift", "support"],
    ascending=[False, False]
).reset_index(drop=True)

filtered_missed_rules = filtered_missed_rules.sort_values(
    ["lift", "support"],
    ascending=[False, False]
).reset_index(drop=True)

In [33]:
pd.set_option("display.max_colwidth", None)

display(
    filtered_answered_rules[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(20)
)

display(
    filtered_missed_rules[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(20)
)

,antecedents,consequents,support,confidence,lift
0,"(day_of_week=Tuesday, previous_calls=medium_previous_calls, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.026389,1.000000,1.113173
1,"(day_of_week=Tuesday, previous_calls=medium_previous_calls, receiver_answer_rate=perfect_receiver_answer_rate)",(call_outcome=answered),0.024722,1.000000,1.113173
2,"(weekend=yes, recent_sms=high_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.020833,1.000000,1.113173
3,"(receiver_answer_rate=no_receiver_history, time_since_last_call=within_1h)",(call_outcome=answered),0.020556,1.000000,1.113173
4,"(receiver_answer_rate=perfect_receiver_answer_rate, previous_answer_rate=high_answer_rate, time_since_last_call=within_1h)",(call_outcome=answered),0.036111,0.992366,1.104675
5,"(recent_sms=high_recent_sms, previous_answer_rate=high_answer_rate, receiver_answer_rate=perfect_receiver_answer_rate)",(call_outcome=answered),0.031667,0.991304,1.103493
6,"(time_since_last_call=1h_to_24h, recent_sms=no_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.030833,0.991071,1.103234
7,"(previous_answer_rate=perfect_answer_rate, recent_sms=medium_recent_sms, bluetooth=medium_recent_proximity)",(call_outcome=answered),0.029444,0.990654,1.102769
8,"(previous_calls=high_previous_calls, day_of_week=Saturday)",(call_outcome=answered),0.027222,0.989899,1.101928
9,"(day_of_week=Wednesday, receiver_answer_rate=perfect_receiver_answer_rate, time_since_last_call=within_1h)",(call_outcome=answered),0.027222,0.989899,1.101928


,antecedents,consequents,support,confidence,lift
0,(receiver_answer_rate=low_receiver_answer_rate),(call_outcome=missed),0.020278,0.374359,3.682219
1,(receiver_answer_rate=medium_receiver_answer_rate),(call_outcome=missed),0.021944,0.201531,1.982268


In [34]:
missed_rules["antecedent_length"] = (
    missed_rules["antecedents"].apply(len)
)

multi_feature_missed = missed_rules[
    missed_rules["antecedent_length"] >= 2
].copy()

multi_feature_missed = multi_feature_missed.sort_values(
    ["lift", "support"],
    ascending=[False, False]
).reset_index(drop=True)

print(
    "Missed rules with at least two antecedent items:",
    len(multi_feature_missed)
)

display(
    multi_feature_missed[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift",
            "antecedent_length"
        ]
    ].head(20)
)

Missed rules with at least two antecedent items: 12


,antecedents,consequents,support,confidence,lift,antecedent_length
0,"(time_since_last_sms=no_previous_sms, recent_sms=no_recent_sms)",(call_outcome=missed),0.028889,0.139973,1.376784,2
1,"(previous_answer_rate=no_previous_call_history, previous_calls=no_previous_calls)",(call_outcome=missed),0.021667,0.125604,1.235448,2
2,"(time_since_last_call=no_previous_call, previous_calls=no_previous_calls)",(call_outcome=missed),0.021667,0.125604,1.235448,2
3,"(previous_answer_rate=no_previous_call_history, time_since_last_call=no_previous_call)",(call_outcome=missed),0.021667,0.125604,1.235448,2
4,"(previous_answer_rate=no_previous_call_history, time_since_last_call=no_previous_call, previous_calls=no_previous_calls)",(call_outcome=missed),0.021667,0.125604,1.235448,3
5,"(recent_sms=medium_recent_sms, weekend=no)",(call_outcome=missed),0.020278,0.122278,1.202735,2
6,"(bluetooth=no_recent_proximity, recent_sms=no_recent_sms)",(call_outcome=missed),0.025833,0.121252,1.192639,2
7,"(day_of_week=Friday, weekend=no)",(call_outcome=missed),0.027500,0.117998,1.160632,2
8,"(time_of_day=afternoon, recent_sms=no_recent_sms)",(call_outcome=missed),0.023889,0.113907,1.120400,2
9,"(bluetooth=no_recent_proximity, weekend=no)",(call_outcome=missed),0.035000,0.112601,1.107546,2


In [35]:
def is_redundant_rule(antecedents):
    items = set(antecedents)

    redundant_groups = [
        {
            "previous_calls=no_previous_calls",
            "previous_answer_rate=no_previous_call_history"
        },
        {
            "previous_calls=no_previous_calls",
            "time_since_last_call=no_previous_call"
        },
        {
            "previous_answer_rate=no_previous_call_history",
            "time_since_last_call=no_previous_call"
        },
        {
            "recent_sms=no_recent_sms",
            "time_since_last_sms=no_previous_sms"
        }
    ]

    # Day of week already determines weekend status
    if any(item.startswith("day_of_week=") for item in items):
        if "weekend=yes" in items or "weekend=no" in items:
            return True

    for group in redundant_groups:
        if group.issubset(items):
            return True

    return False


clean_multi_feature_missed = multi_feature_missed[
    ~multi_feature_missed["antecedents"].apply(is_redundant_rule)
].copy()

clean_multi_feature_missed = clean_multi_feature_missed.sort_values(
    ["lift", "support"],
    ascending=[False, False]
).reset_index(drop=True)

display(
    clean_multi_feature_missed[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ]
)

,antecedents,consequents,support,confidence,lift
0,"(recent_sms=medium_recent_sms, weekend=no)",(call_outcome=missed),0.020278,0.122278,1.202735
1,"(bluetooth=no_recent_proximity, recent_sms=no_recent_sms)",(call_outcome=missed),0.025833,0.121252,1.192639
2,"(time_of_day=afternoon, recent_sms=no_recent_sms)",(call_outcome=missed),0.023889,0.113907,1.120400
3,"(bluetooth=no_recent_proximity, weekend=no)",(call_outcome=missed),0.035000,0.112601,1.107546
4,"(time_of_day=afternoon, bluetooth=no_recent_proximity)",(call_outcome=missed),0.023611,0.107868,1.060997
5,"(previous_calls=medium_previous_calls, weekend=no)",(call_outcome=missed),0.023889,0.103865,1.021620


In [36]:
answered_rules["antecedent_length"] = (
    answered_rules["antecedents"].apply(len)
)

multi_feature_answered = answered_rules[
    answered_rules["antecedent_length"] >= 2
].copy()

clean_multi_feature_answered = multi_feature_answered[
    ~multi_feature_answered["antecedents"].apply(is_redundant_rule)
].copy()

clean_multi_feature_answered = clean_multi_feature_answered.sort_values(
    ["lift", "support"],
    ascending=[False, False]
).reset_index(drop=True)

display(
    clean_multi_feature_answered[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20)
)

,antecedents,consequents,support,confidence,lift
0,"(day_of_week=Tuesday, previous_calls=medium_previous_calls, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.026389,1.000000,1.113173
1,"(day_of_week=Tuesday, previous_calls=medium_previous_calls, receiver_answer_rate=perfect_receiver_answer_rate)",(call_outcome=answered),0.024722,1.000000,1.113173
2,"(weekend=yes, recent_sms=high_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.020833,1.000000,1.113173
3,"(receiver_answer_rate=no_receiver_history, time_since_last_call=within_1h)",(call_outcome=answered),0.020556,1.000000,1.113173
4,"(receiver_answer_rate=perfect_receiver_answer_rate, previous_answer_rate=high_answer_rate, time_since_last_call=within_1h)",(call_outcome=answered),0.036111,0.992366,1.104675
5,"(recent_sms=high_recent_sms, previous_answer_rate=high_answer_rate, receiver_answer_rate=perfect_receiver_answer_rate)",(call_outcome=answered),0.031667,0.991304,1.103493
6,"(time_since_last_call=1h_to_24h, recent_sms=no_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.030833,0.991071,1.103234
7,"(previous_answer_rate=perfect_answer_rate, recent_sms=medium_recent_sms, bluetooth=medium_recent_proximity)",(call_outcome=answered),0.029444,0.990654,1.102769
8,"(previous_calls=high_previous_calls, day_of_week=Saturday)",(call_outcome=answered),0.027222,0.989899,1.101928
9,"(day_of_week=Wednesday, receiver_answer_rate=perfect_receiver_answer_rate, time_since_last_call=within_1h)",(call_outcome=answered),0.027222,0.989899,1.101928


In [37]:
# Count how many clean multi-feature answered rules remain
print("Clean multi-feature answered rules:", len(clean_multi_feature_answered))

clean_multi_feature_answered_readable = clean_multi_feature_answered.copy()

clean_multi_feature_answered_readable["antecedents"] = (
    clean_multi_feature_answered_readable["antecedents"]
    .apply(lambda x: ", ".join(sorted(list(x))))
)

clean_multi_feature_answered_readable["consequents"] = (
    clean_multi_feature_answered_readable["consequents"]
    .apply(lambda x: ", ".join(sorted(list(x))))
)

display(
    clean_multi_feature_answered_readable[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20)
)

Clean multi-feature answered rules: 1224


,antecedents,consequents,support,confidence,lift
0,"day_of_week=Tuesday, previous_answer_rate=perfect_answer_rate, previous_calls=medium_previous_calls",call_outcome=answered,0.026389,1.000000,1.113173
1,"day_of_week=Tuesday, previous_calls=medium_previous_calls, receiver_answer_rate=perfect_receiver_answer_rate",call_outcome=answered,0.024722,1.000000,1.113173
2,"previous_answer_rate=perfect_answer_rate, recent_sms=high_recent_sms, weekend=yes",call_outcome=answered,0.020833,1.000000,1.113173
3,"receiver_answer_rate=no_receiver_history, time_since_last_call=within_1h",call_outcome=answered,0.020556,1.000000,1.113173
4,"previous_answer_rate=high_answer_rate, receiver_answer_rate=perfect_receiver_answer_rate, time_since_last_call=within_1h",call_outcome=answered,0.036111,0.992366,1.104675
5,"previous_answer_rate=high_answer_rate, receiver_answer_rate=perfect_receiver_answer_rate, recent_sms=high_recent_sms",call_outcome=answered,0.031667,0.991304,1.103493
6,"previous_answer_rate=perfect_answer_rate, recent_sms=no_recent_sms, time_since_last_call=1h_to_24h",call_outcome=answered,0.030833,0.991071,1.103234
7,"bluetooth=medium_recent_proximity, previous_answer_rate=perfect_answer_rate, recent_sms=medium_recent_sms",call_outcome=answered,0.029444,0.990654,1.102769
8,"day_of_week=Saturday, previous_calls=high_previous_calls",call_outcome=answered,0.027222,0.989899,1.101928
9,"day_of_week=Wednesday, receiver_answer_rate=perfect_receiver_answer_rate, time_since_last_call=within_1h",call_outcome=answered,0.027222,0.989899,1.101928


Remove overly specific / redundant answered rules

In [38]:
def remove_redundant_rules(rule_df, lift_tolerance=0.01):
    
    keep = []

    for i, row_i in rule_df.iterrows():
        
        antecedent_i = set(row_i["antecedents"])
        redundant = False

        for j, row_j in rule_df.iterrows():

            if i == j:
                continue

            antecedent_j = set(row_j["antecedents"])

            # Check if another rule has a simpler antecedent
            # contained in the current rule
            if (
                antecedent_j < antecedent_i
                and row_j["consequents"] == row_i["consequents"]
                and row_j["lift"] >= row_i["lift"] - lift_tolerance
            ):
                redundant = True
                break

        if not redundant:
            keep.append(i)

    return rule_df.loc[keep].copy()


representative_answered_rules = remove_redundant_rules(
    clean_multi_feature_answered
)

representative_answered_rules = (
    representative_answered_rules
    .sort_values(
        ["lift", "support"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

print(
    "Representative answered rules:",
    len(representative_answered_rules)
)

Representative answered rules: 483


In [39]:
display(
    representative_answered_rules[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20)
)

,antecedents,consequents,support,confidence,lift
0,"(day_of_week=Tuesday, previous_calls=medium_previous_calls, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.026389,1.000000,1.113173
1,"(day_of_week=Tuesday, previous_calls=medium_previous_calls, receiver_answer_rate=perfect_receiver_answer_rate)",(call_outcome=answered),0.024722,1.000000,1.113173
2,"(weekend=yes, recent_sms=high_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.020833,1.000000,1.113173
3,"(receiver_answer_rate=no_receiver_history, time_since_last_call=within_1h)",(call_outcome=answered),0.020556,1.000000,1.113173
4,"(time_since_last_call=1h_to_24h, recent_sms=no_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.030833,0.991071,1.103234
5,"(previous_answer_rate=perfect_answer_rate, recent_sms=medium_recent_sms, bluetooth=medium_recent_proximity)",(call_outcome=answered),0.029444,0.990654,1.102769
6,"(previous_calls=high_previous_calls, day_of_week=Saturday)",(call_outcome=answered),0.027222,0.989899,1.101928
7,"(day_of_week=Wednesday, receiver_answer_rate=perfect_receiver_answer_rate, time_since_last_call=within_1h)",(call_outcome=answered),0.027222,0.989899,1.101928
8,"(time_since_last_call=1h_to_24h, recent_sms=no_recent_sms, receiver_answer_rate=perfect_receiver_answer_rate)",(call_outcome=answered),0.026944,0.989796,1.101814
9,"(time_of_day=evening, recent_sms=medium_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.026667,0.989691,1.101697


Select top representative answered rules

In [40]:
top_answered_rules = representative_answered_rules[
    representative_answered_rules["support"] >= 0.02
].copy()

top_answered_rules = top_answered_rules.sort_values(
    ["lift", "support"],
    ascending=[False, False]
).reset_index(drop=True)

display(
    top_answered_rules[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(15)
)

,antecedents,consequents,support,confidence,lift
0,"(day_of_week=Tuesday, previous_calls=medium_previous_calls, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.026389,1.000000,1.113173
1,"(day_of_week=Tuesday, previous_calls=medium_previous_calls, receiver_answer_rate=perfect_receiver_answer_rate)",(call_outcome=answered),0.024722,1.000000,1.113173
2,"(weekend=yes, recent_sms=high_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.020833,1.000000,1.113173
3,"(receiver_answer_rate=no_receiver_history, time_since_last_call=within_1h)",(call_outcome=answered),0.020556,1.000000,1.113173
4,"(time_since_last_call=1h_to_24h, recent_sms=no_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.030833,0.991071,1.103234
5,"(previous_answer_rate=perfect_answer_rate, recent_sms=medium_recent_sms, bluetooth=medium_recent_proximity)",(call_outcome=answered),0.029444,0.990654,1.102769
6,"(previous_calls=high_previous_calls, day_of_week=Saturday)",(call_outcome=answered),0.027222,0.989899,1.101928
7,"(day_of_week=Wednesday, receiver_answer_rate=perfect_receiver_answer_rate, time_since_last_call=within_1h)",(call_outcome=answered),0.027222,0.989899,1.101928
8,"(time_since_last_call=1h_to_24h, recent_sms=no_recent_sms, receiver_answer_rate=perfect_receiver_answer_rate)",(call_outcome=answered),0.026944,0.989796,1.101814
9,"(time_of_day=evening, recent_sms=medium_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.026667,0.989691,1.101697


Final summary of representative association rules

In [41]:
final_answered = representative_answered_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].head(5).copy()

final_missed = clean_multi_feature_missed[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].head(5).copy()

final_rules = pd.concat(
    [final_answered, final_missed],
    ignore_index=True
)

display(final_rules)

,antecedents,consequents,support,confidence,lift
0,"(day_of_week=Tuesday, previous_calls=medium_previous_calls, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.026389,1.000000,1.113173
1,"(day_of_week=Tuesday, previous_calls=medium_previous_calls, receiver_answer_rate=perfect_receiver_answer_rate)",(call_outcome=answered),0.024722,1.000000,1.113173
2,"(weekend=yes, recent_sms=high_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.020833,1.000000,1.113173
3,"(receiver_answer_rate=no_receiver_history, time_since_last_call=within_1h)",(call_outcome=answered),0.020556,1.000000,1.113173
4,"(time_since_last_call=1h_to_24h, recent_sms=no_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.030833,0.991071,1.103234
5,"(recent_sms=medium_recent_sms, weekend=no)",(call_outcome=missed),0.020278,0.122278,1.202735
6,"(bluetooth=no_recent_proximity, recent_sms=no_recent_sms)",(call_outcome=missed),0.025833,0.121252,1.192639
7,"(time_of_day=afternoon, recent_sms=no_recent_sms)",(call_outcome=missed),0.023889,0.113907,1.120400
8,"(bluetooth=no_recent_proximity, weekend=no)",(call_outcome=missed),0.035000,0.112601,1.107546
9,"(time_of_day=afternoon, bluetooth=no_recent_proximity)",(call_outcome=missed),0.023611,0.107868,1.060997


Final representative association rules

In [42]:
def select_rule(rule_df, antecedent_items):
    target = frozenset(antecedent_items)

    result = rule_df[
        rule_df["antecedents"] == target
    ]

    return result


# Representative answered-call patterns
answered_selections = [
    [
        "weekend=yes",
        "recent_sms=high_recent_sms",
        "previous_answer_rate=perfect_answer_rate"
    ],
    [
        "bluetooth=medium_recent_proximity",
        "previous_answer_rate=perfect_answer_rate",
        "recent_sms=medium_recent_sms"
    ],
    [
        "previous_answer_rate=high_answer_rate",
        "receiver_answer_rate=perfect_receiver_answer_rate",
        "time_since_last_call=within_1h"
    ],
    [
        "previous_calls=high_previous_calls",
        "day_of_week=Saturday"
    ],
    [
        "time_of_day=evening",
        "previous_answer_rate=perfect_answer_rate",
        "recent_sms=medium_recent_sms"
    ]
]


# Representative missed-call patterns
missed_selections = [
    [
        "receiver_answer_rate=low_receiver_answer_rate"
    ],
    [
        "receiver_answer_rate=medium_receiver_answer_rate"
    ],
    [
        "bluetooth=no_recent_proximity",
        "recent_sms=no_recent_sms"
    ],
    [
        "time_of_day=afternoon",
        "recent_sms=no_recent_sms"
    ],
    [
        "weekend=no",
        "bluetooth=no_recent_proximity"
    ]
]


selected_answered = pd.concat(
    [
        select_rule(answered_rules, rule)
        for rule in answered_selections
    ],
    ignore_index=True
)

selected_missed = pd.concat(
    [
        select_rule(missed_rules, rule)
        for rule in missed_selections
    ],
    ignore_index=True
)

final_representative_rules = pd.concat(
    [selected_answered, selected_missed],
    ignore_index=True
)

display(
    final_representative_rules[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ]
)

,antecedents,consequents,support,confidence,lift
0,"(weekend=yes, recent_sms=high_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.020833,1.000000,1.113173
1,"(previous_answer_rate=perfect_answer_rate, recent_sms=medium_recent_sms, bluetooth=medium_recent_proximity)",(call_outcome=answered),0.029444,0.990654,1.102769
2,"(receiver_answer_rate=perfect_receiver_answer_rate, previous_answer_rate=high_answer_rate, time_since_last_call=within_1h)",(call_outcome=answered),0.036111,0.992366,1.104675
3,"(previous_calls=high_previous_calls, day_of_week=Saturday)",(call_outcome=answered),0.027222,0.989899,1.101928
4,"(time_of_day=evening, recent_sms=medium_recent_sms, previous_answer_rate=perfect_answer_rate)",(call_outcome=answered),0.026667,0.989691,1.101697
5,(receiver_answer_rate=low_receiver_answer_rate),(call_outcome=missed),0.020278,0.374359,3.682219
6,(receiver_answer_rate=medium_receiver_answer_rate),(call_outcome=missed),0.021944,0.201531,1.982268
7,"(bluetooth=no_recent_proximity, recent_sms=no_recent_sms)",(call_outcome=missed),0.025833,0.121252,1.192639
8,"(time_of_day=afternoon, recent_sms=no_recent_sms)",(call_outcome=missed),0.023889,0.113907,1.120400
9,"(bluetooth=no_recent_proximity, weekend=no)",(call_outcome=missed),0.035000,0.112601,1.107546


Save assosiation rules

In [43]:
final_representative_rules.to_csv(
    "../data/association_rules_representative.csv",
    index=False
)

outcome_rules.to_csv(
    "../data/association_rules_all_outcomes.csv",
    index=False
)

print("Association rule results saved.")

Association rule results saved.
